# Step 1 — Railway Telemetry Dataset

Generates the synthetic railway track telemetry dataset (`railway_telemetry.csv`) and syncs it with **Vayu Object Storage (S3)**.

| Section | What it does |
|---|---|
| 1. Setup | Loads S3 credentials from the root `.env` and creates the S3 client |
| 2. Generate | Creates `railway_telemetry.csv` (2,000 rows) — **overwrites** the local file |
| 3. Upload | Uploads the CSV to `s3://$S3_BUCKET_NAME/$S3_DATASET_KEY` |
| 4. Download | Pulls the CSV back from S3 (use this instead of section 2 if the dataset already exists in your bucket) |

## 1. Setup

In [ ]:
import os
import random
from datetime import datetime, timedelta
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
from botocore.config import Config
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env", override=True)

os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"] = "when_required"

AWS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET = os.getenv("AWS_SECRET_ACCESS_KEY")
S3_ENDPOINT = os.getenv("S3_ENDPOINT")
BUCKET = os.getenv("S3_BUCKET_NAME")
S3_KEY = os.getenv("S3_DATASET_KEY", "railway/railway_telemetry.csv")
LOCAL_FILE = "railway_telemetry.csv"

missing = [name for name, value in {
    "AWS_ACCESS_KEY_ID": AWS_KEY,
    "AWS_SECRET_ACCESS_KEY": AWS_SECRET,
    "S3_ENDPOINT": S3_ENDPOINT,
    "S3_BUCKET_NAME": BUCKET,
}.items() if not value]
if missing:
    raise ValueError(f"Missing in .env: {missing}")

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)
print(f"S3 client ready — bucket: {BUCKET}, key: {S3_KEY}")

## 2. Generate the dataset

Three operating regimes are simulated — normal (75%), high wear (15%) and emergency (10%). `failure_probability` is a weighted score of vibration, rail strain, alignment deviation and temperature plus a little noise; `risk_level`, `fault_type` and `recommended_action` are derived from it.

In [ ]:
def generate_synthetic_rail_data(output_path, num_samples=2000):
    print("🚆 Generating railway telemetry dataset...")

    np.random.seed(42)
    random.seed(42)

    # -----------------------------
    # Generate sensor data
    # -----------------------------

    num_normal = int(num_samples * 0.75)
    num_vuln = int(num_samples * 0.15)
    num_emergency = num_samples - num_normal - num_vuln

    # Normal tracks
    df_normal = pd.DataFrame({
        "vibration_hz": np.random.normal(50, 5, num_normal),
        "acoustic_emission_db": np.random.normal(40, 3, num_normal),
        "rail_strain_mu": np.random.normal(120, 10, num_normal),
        "track_temperature_c": np.random.normal(32, 4, num_normal),
        "alignment_deviation_mm": np.random.normal(0, 0.5, num_normal),
    })

    # High wear
    df_vuln = pd.DataFrame({
        "vibration_hz": np.random.normal(68, 4, num_vuln),
        "acoustic_emission_db": np.random.normal(52, 4, num_vuln),
        "rail_strain_mu": np.random.normal(142, 8, num_vuln),
        "track_temperature_c": np.random.normal(48, 3, num_vuln),
        "alignment_deviation_mm": np.random.normal(2.1, 0.8, num_vuln),
    })

    # Emergency
    df_emergency = pd.DataFrame({
        "vibration_hz": np.random.normal(85, 8, num_emergency),
        "acoustic_emission_db": np.random.normal(65, 6, num_emergency),
        "rail_strain_mu": np.random.normal(175, 15, num_emergency),
        "track_temperature_c": np.random.normal(62, 5, num_emergency),
        "alignment_deviation_mm": np.random.choice(
            [
                np.random.normal(-4.5, 0.5),
                np.random.normal(4.5, 0.5),
            ],
            num_emergency,
        ),
    })

    df = pd.concat([df_normal, df_vuln, df_emergency], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    # -------------------------------------------------------
    # Generate failure probability instead of 0/1 prediction
    # -------------------------------------------------------

    score = (
        0.35 * (df["vibration_hz"] / 90)
        + 0.30 * (df["rail_strain_mu"] / 180)
        + 0.20 * (abs(df["alignment_deviation_mm"]) / 5)
        + 0.15 * (df["track_temperature_c"] / 70)
    )

    score = np.clip(score, 0, 1)

    noise = np.random.normal(0, 0.04, len(df))

    df["failure_probability"] = np.clip(score + noise, 0, 1).round(2)

    # -----------------------------
    # Risk Level
    # -----------------------------

    def risk(prob):
        if prob < 0.30:
            return "LOW"
        elif prob < 0.60:
            return "MEDIUM"
        elif prob < 0.85:
            return "HIGH"
        else:
            return "CRITICAL"

    df["risk_level"] = df["failure_probability"].apply(risk)

    # -----------------------------
    # Track metadata
    # -----------------------------

    track_ids = [f"TRK-{i:03d}" for i in range(1, 51)]

    df["track_id"] = np.random.choice(track_ids, len(df))

    df["section"] = np.random.choice(
        ["North", "South", "East", "West"],
        len(df),
    )

    # -----------------------------
    # Timestamp
    # -----------------------------

    start = datetime.now() - timedelta(days=2)

    df["timestamp"] = [
        start + timedelta(seconds=i * 60)
        for i in range(len(df))
    ]

    # -----------------------------
    # Fault Type
    # -----------------------------

    def fault(row):

        if row["risk_level"] == "LOW":
            return "Normal"

        if abs(row["alignment_deviation_mm"]) > 4:
            return "Track Misalignment"

        if row["vibration_hz"] > 80:
            return "Rail Crack"

        if row["rail_strain_mu"] > 160:
            return "Structural Stress"

        return "Excessive Wear"

    df["fault_type"] = df.apply(fault, axis=1)

    # -----------------------------
    # Inspection Status
    # -----------------------------

    status = []

    for r in df["risk_level"]:

        if r == "LOW":
            status.append("Completed")

        elif r == "MEDIUM":
            status.append(random.choice([
                "Completed",
                "Pending"
            ]))

        elif r == "HIGH":
            status.append(random.choice([
                "Pending",
                "In Progress"
            ]))

        else:
            status.append(random.choice([
                "Pending",
                "In Progress"
            ]))

    df["inspection_status"] = status

    # -----------------------------
    # Recommended Action
    # -----------------------------

    def action(r):

        if r == "LOW":
            return "Monitor"

        elif r == "MEDIUM":
            return "Schedule Inspection"

        elif r == "HIGH":
            return "Repair Soon"

        else:
            return "Immediate Shutdown"

    df["recommended_action"] = df["risk_level"].apply(action)

    # -----------------------------
    # Assigned Team
    # -----------------------------

    df["assigned_team"] = np.random.choice(
        [
            "Team Alpha",
            "Team Bravo",
            "Team Charlie",
            "Unassigned",
        ],
        len(df),
        p=[0.3, 0.3, 0.2, 0.2],
    )

    # -----------------------------
    # Maintenance
    # -----------------------------

    df["last_maintenance_days"] = np.random.randint(
        1,
        180,
        len(df),
    )

    df["action_taken"] = np.where(
        df["inspection_status"] == "Completed",
        "Yes",
        "No",
    )

    # -----------------------------
    # Reorder columns
    # -----------------------------

    df = df[
        [
            "timestamp",
            "track_id",
            "section",
            "vibration_hz",
            "acoustic_emission_db",
            "rail_strain_mu",
            "track_temperature_c",
            "alignment_deviation_mm",
            "failure_probability",
            "risk_level",
            "fault_type",
            "inspection_status",
            "recommended_action",
            "assigned_team",
            "last_maintenance_days",
            "action_taken",
        ]
    ]

    df.to_csv(output_path, index=False)

    print(f"✅ Generated {len(df)} records")
    print(f"📁 Saved to {output_path}")


generate_synthetic_rail_data(LOCAL_FILE)

## 3. Upload to Vayu Object Storage

In [ ]:
s3.upload_file(Filename=LOCAL_FILE, Bucket=BUCKET, Key=S3_KEY)
print(f"Uploaded {LOCAL_FILE} -> s3://{BUCKET}/{S3_KEY}")

## 4. Download from Vayu Object Storage

In [ ]:
s3.download_file(BUCKET, S3_KEY, LOCAL_FILE)
print(f"Downloaded s3://{BUCKET}/{S3_KEY} -> {LOCAL_FILE}")
pd.read_csv(LOCAL_FILE).head()